# PV 발전실적 EDA
남동발전 태양광 시간대별 발전실적 데이터 탐색

**데이터:** 2022.01 ~ 2025.12 (48개월, 15개 사이트, 23개 호기)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
import os
from pathlib import Path

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

DATA_DIR = Path("../../data/solar_hourly")
print(f"데이터 경로: {DATA_DIR.resolve()}")
print(f"CSV 파일: {len(list(DATA_DIR.glob('solar_*.csv')))}개")

## 1. 데이터 로딩 및 정리

In [ ]:
# 모든 solar CSV 로드 & 합치기 (이미 utf-8-sig로 변환됨)
dfs = []
for f in sorted(DATA_DIR.glob("solar_*.csv")):
    df = pd.read_csv(f, encoding='utf-8-sig', index_col=False)
    df.columns = df.columns.str.strip()
    dfs.append(df)

raw = pd.concat(dfs, ignore_index=True)
print(f"전체: {len(raw)} rows, {len(raw.columns)} cols")
print(f"컬럼: {raw.columns.tolist()}")

In [ ]:
# 컬럼명 정리
col_map = {}
for c in raw.columns:
    if '발전' in c and '량' not in c and 'KWh' not in c and 'KW' not in c: col_map[c] = 'site'
    elif '호기' in c: col_map[c] = 'unit'
    elif '일자' in c or '날짜' in c: col_map[c] = 'date'
    elif '총량' in c or '합계' in c: col_map[c] = 'total_kw'
    elif '평균' in c and 'KW' in c: col_map[c] = 'avg_kw'
    elif '최대(시간별)' in c or '최대(시간대)' in c: col_map[c] = 'max_hour'
    elif '최소(시간별)' in c or '최소(시간대)' in c: col_map[c] = 'min_hour'

# 시간대 컬럼
hour_cols_orig = [c for c in raw.columns if 'KWh' in c]
for c in hour_cols_orig:
    h = int(c.split('시')[0].replace('발전량', '').strip())
    col_map[c] = f'h{h:02d}'

raw = raw.rename(columns=col_map)

# 타입 정리
raw['site'] = raw['site'].astype(str).str.strip()
raw['unit'] = raw['unit'].astype(int)
raw['date'] = pd.to_datetime(raw['date'].astype(str).str.strip())

hour_cols = [f'h{h:02d}' for h in range(1, 25)]
for c in hour_cols:
    raw[c] = pd.to_numeric(raw[c], errors='coerce').fillna(0)

raw['site_unit'] = raw['site'] + '_' + raw['unit'].astype(str)

print(f"정리 완료: {raw.shape}")
print(f"기간: {raw['date'].min()} ~ {raw['date'].max()}")
print(f"사이트: {raw['site'].nunique()}개, 사이트_호기: {raw['site_unit'].nunique()}개")
raw.head(3)

## 2. 사이트별 기본 현황

In [ ]:
# 사이트별 일 총발전량 계산
raw['daily_kwh'] = raw[hour_cols].sum(axis=1)

# 사이트별 요약
site_summary = raw.groupby('site_unit').agg(
    days=('date', 'nunique'),
    mean_kwh=('daily_kwh', 'mean'),
    max_kwh=('daily_kwh', 'max'),
    min_kwh=('daily_kwh', 'min'),
    std_kwh=('daily_kwh', 'std'),
).sort_values('mean_kwh', ascending=False)

site_summary['mean_mwh'] = site_summary['mean_kwh'] / 1000
site_summary['cv'] = site_summary['std_kwh'] / site_summary['mean_kwh']  # 변동계수

print("=== 사이트별 일 발전량 요약 ===")
print(site_summary.to_string())
print(f"\n포트폴리오 일평균: {site_summary['mean_kwh'].sum()/1000:.1f} MWh")

In [ ]:
# 사이트별 일평균 발전량 바차트
fig, ax = plt.subplots(figsize=(14, 6))
site_summary_sorted = site_summary.sort_values('mean_mwh')
colors = ['#2196F3' if v > 10 else '#90CAF9' if v > 1 else '#BBDEFB' 
          for v in site_summary_sorted['mean_mwh']]
ax.barh(site_summary_sorted.index, site_summary_sorted['mean_mwh'], color=colors)
ax.set_xlabel('일평균 발전량 (MWh)')
ax.set_title('사이트별 일평균 발전량 (2026 2~3월)')
for i, (idx, row) in enumerate(site_summary_sorted.iterrows()):
    ax.text(row['mean_mwh'] + 0.5, i, f"{row['mean_mwh']:.1f}", va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 3. 시간대별 발전 패턴 (태양광 커브)

In [ ]:
# 포트폴리오 전체 시간대별 합산
hourly_total = raw.groupby('date')[hour_cols].sum()  # 날짜별, 시간대별 전체 합산 (KWh)

# 평균 일 프로파일
mean_profile = hourly_total.mean() / 1000  # MWh
std_profile = hourly_total.std() / 1000

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# (좌) 평균 프로파일 + 표준편차 밴드
hours = range(1, 25)
ax = axes[0]
ax.plot(hours, mean_profile.values, 'o-', color='#FF6F00', linewidth=2, label='평균')
ax.fill_between(hours, 
                (mean_profile - std_profile).values, 
                (mean_profile + std_profile).values, 
                alpha=0.2, color='#FF6F00', label='±1 std')
ax.set_xlabel('시간')
ax.set_ylabel('발전량 (MWh)')
ax.set_title('포트폴리오 전체 - 시간대별 평균 발전량')
ax.set_xticks(range(1, 25))
ax.legend()
ax.grid(True, alpha=0.3)

# (우) 모든 날의 프로파일을 겹쳐서 (스파게티 플롯)
ax = axes[1]
for _, row in hourly_total.iterrows():
    ax.plot(hours, row.values / 1000, alpha=0.15, color='#1976D2', linewidth=0.8)
ax.plot(hours, mean_profile.values, color='#FF6F00', linewidth=2.5, label='평균')
ax.set_xlabel('시간')
ax.set_ylabel('발전량 (MWh)')
ax.set_title('일별 발전 프로파일 (모든 날 겹침)')
ax.set_xticks(range(1, 25))
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 주요 사이트별 시간대 프로파일 비교
top_sites = site_summary.nlargest(6, 'mean_kwh').index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, su in enumerate(top_sites):
    ax = axes[i]
    sub = raw[raw['site_unit'] == su]
    
    for _, row in sub.iterrows():
        ax.plot(hours, [row[f'h{h:02d}'] for h in range(1, 25)], 
                alpha=0.2, color='#1976D2', linewidth=0.7)
    
    mean_vals = sub[hour_cols].mean()
    ax.plot(hours, mean_vals.values, color='#FF6F00', linewidth=2, label='평균')
    
    ax.set_title(f'{su}\n(일평균 {site_summary.loc[su, "mean_mwh"]:.1f} MWh)', fontsize=10)
    ax.set_xticks([6, 9, 12, 15, 18, 21])
    ax.grid(True, alpha=0.3)

fig.suptitle('상위 6개 사이트 - 시간대별 발전 패턴', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 4. 일별 총발전량 추이 & 변동성

In [ ]:
# 포트폴리오 전체 일별 총발전량
daily_total = raw.groupby('date')['daily_kwh'].sum() / 1000  # MWh

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# (상) 일별 총발전량 시계열
ax = axes[0]
ax.bar(daily_total.index, daily_total.values, color='#FFA726', alpha=0.8, width=0.8)
ax.axhline(daily_total.mean(), color='red', linestyle='--', linewidth=1.5, label=f'평균 {daily_total.mean():.0f} MWh')
ax.set_ylabel('일 총발전량 (MWh)')
ax.set_title('포트폴리오 일별 총발전량')
ax.legend()
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))  # 매주 월요일

# (하) 전일 대비 변화율
daily_change = daily_total.pct_change() * 100
ax = axes[1]
colors = ['#E53935' if abs(v) > 30 else '#FF7043' if abs(v) > 15 else '#66BB6A' 
          for v in daily_change.dropna()]
ax.bar(daily_change.dropna().index, daily_change.dropna().values, color=colors, alpha=0.8, width=0.8)
ax.axhline(0, color='black', linewidth=0.5)
ax.axhline(30, color='red', linestyle=':', alpha=0.5, label='±30% (급변 기준)')
ax.axhline(-30, color='red', linestyle=':', alpha=0.5)
ax.set_ylabel('전일 대비 변화율 (%)')
ax.set_title('일별 발전량 변화율 (빨강=급변)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))

ramp_days = (daily_change.abs() > 30).sum()
print(f"급변일 (전일 대비 ±30% 이상): {ramp_days}일 / {len(daily_change.dropna())}일 ({ramp_days/len(daily_change.dropna())*100:.0f}%)")

plt.tight_layout()
plt.show()

## 5. 히트맵: 날짜 × 시간 발전량

In [ ]:
# 히트맵: 날짜(y) × 시간(x) = 포트폴리오 전체 발전량
heatmap_data = hourly_total / 1000  # MWh
heatmap_data.index = heatmap_data.index.strftime('%m/%d')
heatmap_data.columns = [f'{h}시' for h in range(1, 25)]

fig, ax = plt.subplots(figsize=(16, 12))
sns.heatmap(heatmap_data, cmap='YlOrRd', ax=ax, 
            xticklabels=True, yticklabels=True,
            cbar_kws={'label': '발전량 (MWh)'},
            linewidths=0.1, linecolor='white')
ax.set_title('포트폴리오 전체 - 날짜 × 시간 발전량 히트맵', fontsize=13)
ax.set_xlabel('시간')
ax.set_ylabel('날짜')
plt.tight_layout()
plt.show()

## 6. 사이트 간 상관관계

In [ ]:
# 사이트별 일 총발전량으로 상관행렬
daily_by_site = raw.groupby(['date', 'site_unit'])['daily_kwh'].sum().unstack(fill_value=0)

# 발전량 0인 사이트 제거
active_sites = daily_by_site.columns[daily_by_site.mean() > 100]
daily_active = daily_by_site[active_sites]

corr = daily_active.corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, cmap='RdYlBu_r', center=0.5, vmin=0, vmax=1,
            annot=True, fmt='.2f', square=True, ax=ax,
            linewidths=0.5, cbar_kws={'label': '상관계수'})
ax.set_title('사이트 간 일 발전량 상관관계', fontsize=13)
plt.tight_layout()
plt.show()

print(f"\n평균 상관계수: {corr.values[np.tril_indices_from(corr.values, -1)].mean():.3f}")
print(f"최저 상관 쌍: {corr.values[np.tril_indices_from(corr.values, -1)].min():.3f}")

## 7. 이상치 / 데이터 품질 체크

In [ ]:
# 데이터 품질 체크

print("=== 결측 / 이상치 체크 ===\n")

# 1) 야간(1~6시, 22~24시)에 발전량 > 0인 경우
night_cols = ['h01','h02','h03','h04','h05','h06','h22','h23','h24']
night_nonzero = raw[raw[night_cols].sum(axis=1) > 0]
print(f"1) 야간(1~6시, 22~24시) 발전량 > 0: {len(night_nonzero)}건 / {len(raw)}건")
if len(night_nonzero) > 0:
    print(f"   → 사이트: {night_nonzero['site_unit'].unique()}")
    print(f"   → 야간 발전량 합계 범위: {night_nonzero[night_cols].sum(axis=1).min():.1f} ~ {night_nonzero[night_cols].sum(axis=1).max():.1f} KWh")
    
print()

# 2) 일 총발전량 = 0인 날 (정지/고장?)
zero_days = raw[raw['daily_kwh'] == 0]
print(f"2) 일 총발전량 = 0: {len(zero_days)}건")
if len(zero_days) > 0:
    print(f"   → 사이트: {zero_days['site_unit'].value_counts().to_dict()}")

print()

# 3) 발전량 분포 (boxplot)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
daily_by_site_mwh = daily_active / 1000
daily_by_site_mwh.boxplot(ax=ax, vert=True, rot=90)
ax.set_ylabel('일 발전량 (MWh)')
ax.set_title('사이트별 일 발전량 분포')

ax = axes[1]
# 피크시간(10~16시) 발전량만
peak_cols = [f'h{h:02d}' for h in range(10, 17)]
peak_total = raw.groupby('date')[peak_cols].sum().sum(axis=1) / 1000
peak_total.plot(kind='bar', ax=ax, color='#FF7043', alpha=0.8, width=0.8)
ax.set_ylabel('피크시간 발전량 (MWh)')
ax.set_title('피크시간(10~16시) 일별 발전량')
ax.set_xticklabels([d.strftime('%m/%d') for d in peak_total.index], rotation=90, fontsize=7)
ax.axhline(peak_total.mean(), color='red', linestyle='--', label=f'평균 {peak_total.mean():.0f}')
ax.legend()

plt.tight_layout()
plt.show()

## 8. 좋은 날 vs 나쁜 날 비교

In [ ]:
# 일 총발전량 기준 상위 5일(맑은날) vs 하위 5일(흐린날) 비교
top5_dates = daily_total.nlargest(5).index
bot5_dates = daily_total.nsmallest(5).index

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 맑은 날 (상위 5일)
ax = axes[0]
for d in top5_dates:
    vals = hourly_total.loc[d].values / 1000
    ax.plot(hours, vals, 'o-', linewidth=1.5, alpha=0.8, label=d.strftime('%m/%d'))
ax.set_title(f'맑은 날 (상위 5일, 평균 {daily_total[top5_dates].mean():.0f} MWh)')
ax.set_xlabel('시간')
ax.set_ylabel('발전량 (MWh)')
ax.set_xticks(range(1, 25))
ax.legend()
ax.grid(True, alpha=0.3)

# 흐린 날 (하위 5일)
ax = axes[1]
for d in bot5_dates:
    vals = hourly_total.loc[d].values / 1000
    ax.plot(hours, vals, 'o-', linewidth=1.5, alpha=0.8, label=d.strftime('%m/%d'))
ax.set_title(f'흐린 날 (하위 5일, 평균 {daily_total[bot5_dates].mean():.0f} MWh)')
ax.set_xlabel('시간')
ax.set_ylabel('발전량 (MWh)')
ax.set_xticks(range(1, 25))
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(axes[0].get_ylim())  # 같은 스케일

plt.suptitle('맑은 날 vs 흐린 날 발전 프로파일 비교', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# 최대/최소 비율
print(f"최대일 / 최소일 비율: {daily_total.max() / daily_total.min():.1f}배")
print(f"→ 흐린 날 발전량이 맑은 날의 {daily_total.min()/daily_total.max()*100:.0f}%에 불과")

## 9. Persistence Baseline (어제 = 오늘)

In [ ]:
# Persistence: "내일 = 오늘과 같다"
# 이게 가장 단순한 예측. 이것보다 못하면 모델이 의미없음.

actual = hourly_total.copy()  # 날짜 × 24시간
persist = actual.shift(1)     # 어제 = 오늘 예측

# 겹치는 날짜만
valid = actual.index.intersection(persist.dropna().index)
y_true = actual.loc[valid].values.flatten()
y_pred = persist.loc[valid].values.flatten()

# 주간만 (값 > 0인 시간대만 평가)
mask = y_true > 0
y_true_day = y_true[mask]
y_pred_day = y_pred[mask]

rmse = np.sqrt(np.mean((y_true_day - y_pred_day)**2))
mae = np.mean(np.abs(y_true_day - y_pred_day))
nrmse = rmse / np.mean(y_true_day) * 100
nmae = mae / np.mean(y_true_day) * 100

print("=== Persistence Baseline (어제=오늘) — 주간만 ===")
print(f"  RMSE:  {rmse:,.0f} KWh")
print(f"  nRMSE: {nrmse:.1f}%")
print(f"  MAE:   {mae:,.0f} KWh")
print(f"  nMAE:  {nmae:.1f}%")
print(f"\n→ 이 수치가 우리가 이겨야 할 최소 기준")

# 일별 persistence 오차
daily_err = (actual.loc[valid].sum(axis=1) - persist.loc[valid].sum(axis=1)) / 1000  # MWh 차이

fig, ax = plt.subplots(figsize=(14, 5))
colors = ['#E53935' if abs(v) > daily_total.mean()*0.3 else '#66BB6A' for v in daily_err]
ax.bar(daily_err.index, daily_err.values, color=colors, alpha=0.8, width=0.8)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_ylabel('예측 오차 (MWh)')
ax.set_title(f'Persistence 예측 오차 (어제=오늘) — nRMSE={nrmse:.1f}%')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. 기존 CSV 데이터 확인 (사이트 메타데이터)

In [ ]:
# 신재생에너지 사업현황 — 태양광 사이트 메타데이터
meta = pd.read_csv("../../한국남동발전㈜_(발전공기업 표준) 신재생에너지 사업현황_20221231.csv", 
                    encoding='cp949', index_col=False)
meta.columns = meta.columns.str.strip()

# 컬럼명 확인
print("=== 신재생 사업현황 컬럼 ===")
print(meta.columns.tolist())
print(f"\n전체: {len(meta)} rows")

# 에너지원별 현황
energy_col = meta.columns[2]  # 에너지원
name_col = meta.columns[3]    # 사업명
cap_col = meta.columns[4]     # 용량(kW)

print(f"\n=== 에너지원별 현황 ===")
for etype in meta[energy_col].unique():
    sub = meta[meta[energy_col] == etype]
    total_cap = pd.to_numeric(sub[cap_col], errors='coerce').sum()
    print(f"  {etype}: {len(sub)}건, 합계 {total_cap:,.0f} kW = {total_cap/1000:.1f} MW")

print(f"\n=== 태양광 사이트 목록 ===")
solar = meta[meta[energy_col].str.contains('태양', na=False)]
for _, row in solar.iterrows():
    cap = pd.to_numeric(row[cap_col], errors='coerce')
    print(f"  {row[name_col]:30s}  {cap:>8,.0f} kW  ({cap/1000:.1f} MW)")